# Cal3_S2Stereo

A `Cal3_S2Stereo` represents the calibration model for a stereo camera rig. It assumes that both cameras in the rig share the same intrinsic calibration parameters but are separated by a horizontal baseline. 

This class inherits from the standard 5-parameter `Cal3_S2` model and adds a sixth parameter, the baseline `b`, measured in meters. The calibration matrix $K$ is identical to that of `Cal3_S2` and represents the intrinsics of a single camera from a stereo camera rig:

$$
K = \begin{bmatrix} f_x & s & u_0 \\ 0 & f_y & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

The `uncalibrate` and `calibrate` methods of this class behave identically to those in `Cal3_S2`, converting between intrinsic camera coordinates and sensor coordinates for a single camera. Note that `Cal3_S2Stereo` itself does not perform the complete calibration of a stereo camera; it merely stores all the parameters necessary for the calibration. For instance, the baseline parameter `b` is primarily used by other classes like `StereoCamera` to compute the disparity between the left and right images during 3D projection.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3_S2Stereo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%pip install --quiet gtsam-develop

Note: you may need to restart the kernel to use updated packages.


In [1]:
import gtsam
import numpy as np
from gtsam import Cal3_S2Stereo, Point2, Point3, Pose3, StereoCamera

## Initialization

A `Cal3_S2Stereo` object can be initialized in several ways:

In [3]:
# Default constructor
cal_default = Cal3_S2Stereo()
print(f"Default constructor:\n{cal_default}\n")

# From individual parameters: fx, fy, s, u0, v0, b
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal_params = Cal3_S2Stereo(fx, fy, s, u0, v0, b)
print(f"From parameters (fx, fy, s, u0, v0, b):\n{cal_params}\n")

# From a 6-vector [fx, fy, s, u0, v0, b]
param_vector = np.array([1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5])
cal_vector = Cal3_S2Stereo(param_vector)
print(f"From a 6-vector [fx, fy, s, u0, v0, b]:\n{cal_vector}\n")

# From fov in degrees, image dimensions, and stereo baseline
fov, w, h, b = 100, 600, 500, 0.5
cal_fov = Cal3_S2Stereo(fov, w, h, b)
print(f"From fov, image dimensions, and baseline:\n{cal_fov}\n")

Default constructor:
K: 1 0 0
0 1 0
0 0 1
Baseline: 1


From parameters (fx, fy, s, u0, v0, b):
K: 1500  0.1  320
   0 1600  240
   0    0    1
Baseline: 0.5


From a 6-vector [fx, fy, s, u0, v0, b]:
K: 1500  0.1  320
   0 1600  240
   0    0    1
Baseline: 0.5


From fov, image dimensions, and baseline:
K: 251.73      0    300
     0 251.73    250
     0      0      1
Baseline: 0.5




## Properties

Since `Cal3_S2Stereo` inherits from `Cal3_S2`, it has access to all of its parent's property methods, in addition to its own specific methods, as follows:
- `baseline()`: Returns the horizontal baseline $b$, in meters.

In [4]:
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

print("Calibration object properties:")
print(f"fx: {cal.fx()}")
print(f"fy: {cal.fy()}")
print(f"skew: {cal.skew()}")
print(f"u0: {cal.px()}")
print(f"v0: {cal.py()}")
print(f"Baseline: {cal.baseline()}")
print(f"Principal Point: {cal.principalPoint()}")
print(f"Vector [fx, fy, s, u0, v0, b]: {cal.vector()}")
print(f"Aspect ratio: {cal.aspectRatio()}")
print(f"K matrix:\n{cal.K()}")

Calibration object properties:
fx: 1500.0
fy: 1600.0
skew: 0.1
u0: 320.0
v0: 240.0
Baseline: 0.5
Principal Point: [320. 240.]
Vector [fx, fy, s, u0, v0, b]: [1.5e+03 1.6e+03 1.0e-01 3.2e+02 2.4e+02 5.0e-01]
Aspect ratio: 0.9375
K matrix:
[[1.5e+03 1.0e-01 3.2e+02]
 [0.0e+00 1.6e+03 2.4e+02]
 [0.0e+00 0.0e+00 1.0e+00]]


## Basic Operations

All explanations and example code in this section will use the Bumblebee X (BX-P5G-30C-XC3) stereo camera as a representative model. This camera provides a practical reference for understanding `Cal3_S2Stereo`. Below is an example instantiation of a `Cal3_S2Stereo` calibration model using the parameters from the Bumblebee X.

In [5]:
# A sample Cal3_S2Stereo model based on specs of Bumblebee X (BX-P5G-30C-XC3) with Full resolution
width = 2048    # px
height = 1536   # px
baseline = 0.24 # m
px_sz = 3.45    # um
focal_len = 5.7 # mm
fx = focal_len / px_sz * 1000
fy = focal_len / px_sz * 1000
u0 = width / 2
v0 = height / 2
cal_xc3_BumblebeeX = Cal3_S2Stereo(fx=fx, fy=fy, s=0, u0=u0, v0=v0, b=baseline)

print(f"Model properties:\n{cal_xc3_BumblebeeX}")

Model properties:
K: 1652.17       0    1024
      0 1652.17     768
      0       0       1
Baseline: 0.24



### `calibrate()`

The `calibrate(p)` function converts a 2D point `p` from sensor coordinates $(u, v)$ to intrinsic coordinates $(x, y)$. Please review the user guide for [`Cal3_S2`](./Cal3_S2.ipynb) for a more detailed explanation of this function.

In [10]:
def calibration_demo(cal: Cal3_S2Stereo, row: int, col: int):
  u, v = col + 0.5, row + 0.5   # coordinates of pixel center
  x, y = cal.calibrate([u, v])
  print(f"image[{row}, {col}] -> (u, v)=({round(u, 2)}px, {round(v, 2)}px) -> (x, y)=({round(x, 5)}, {round(y, 5)})")
  return x, y

calibration_demo(cal_xc3_BumblebeeX, 0, 0);
calibration_demo(cal_xc3_BumblebeeX, 768, 1024);
calibration_demo(cal_xc3_BumblebeeX, 1536, 2048);

image[0, 0] -> (u, v)=(0.5px, 0.5px) -> (x, y)=(-0.61949, -0.46454)
image[768, 1024] -> (u, v)=(1024.5px, 768.5px) -> (x, y)=(0.0003, 0.0003)
image[1536, 2048] -> (u, v)=(2048.5px, 1536.5px) -> (x, y)=(0.62009, 0.46514)


`calibrate(p)` can also optionally compute Jacobians.

In [7]:
Dcal_calibrate = np.zeros((2, 6), order='F') # Note shape is 2x6
Dp_calibrate = np.zeros((2, 2), order='F')
p_pixels = [768, 1024]
_ = cal.calibrate(p_pixels, Dcal_calibrate, Dp_calibrate)
print(f"\nJacobian Dcal_calibrate:\n{Dcal_calibrate}")
print(f"\nJacobian Dp_calibrate:\n{Dp_calibrate}")


Jacobian Dcal_calibrate:
[[-1.99089333e-04  2.04166667e-08 -3.26666667e-04 -6.66666667e-04
   4.16666667e-08  0.00000000e+00]
 [ 0.00000000e+00 -3.06250000e-04  0.00000000e+00  0.00000000e+00
  -6.25000000e-04  0.00000000e+00]]

Jacobian Dp_calibrate:
[[ 6.66666667e-04 -4.16666667e-08]
 [ 0.00000000e+00  6.25000000e-04]]


### `uncalibrate()`

The `uncalibrate(p)` method converts a 2D point `p` from intrinsic coordinates $(x,y)$ back to sensor coordinates $(u, v)$. Please review the user guide for [`Cal3_S2`](./Cal3_S2.ipynb) for a more detailed explanation of this function.

In [11]:
def uncalibration_demo(cal: Cal3_S2Stereo, x: float, y: float):
  u, v = cal.uncalibrate([x, y])
  print(f"(x, y)=({round(x, 5)}, {round(y, 5)}) -> (u, v)=({round(u, 2)}px, {round(v, 2)}px) -> image[{np.floor(v)}, {np.floor(u)}]")
  return u, v

x, y = calibration_demo(cal_xc3_BumblebeeX, 0, 0);
uncalibration_demo(cal_xc3_BumblebeeX, x, y);
x, y = calibration_demo(cal_xc3_BumblebeeX, 1024, 768);
uncalibration_demo(cal_xc3_BumblebeeX, x, y);

image[0, 0] -> (u, v)=(0.5px, 0.5px) -> (x, y)=(-0.61949, -0.46454)
(x, y)=(-0.61949, -0.46454) -> (u, v)=(0.5px, 0.5px) -> image[0.0, 0.0]
image[1024, 768] -> (u, v)=(768.5px, 1024.5px) -> (x, y)=(-0.15464, 0.15525)
(x, y)=(-0.15464, 0.15525) -> (u, v)=(768.5px, 1024.5px) -> image[1024.0, 768.0]


Likewise, `uncalibrate(p)` can optionally compute Jacobians as well.

In [12]:
# This method can also optionally compute Jacobians.
Dcal_uncalibrate = np.zeros((2, 6), order='F')
Dp_uncalibrate = np.zeros((2, 2), order='F')
p_intrinsic = [0, 0]
_ = cal.uncalibrate(p_intrinsic, Dcal_uncalibrate, Dp_uncalibrate)
print(f"\nJacobian Dcal_uncalibrate:\n{Dcal_uncalibrate}")
print(f"\nJacobian Dp_uncalibrate:\n{Dp_uncalibrate}")


Jacobian Dcal_uncalibrate:
[[0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]]

Jacobian Dp_uncalibrate:
[[1.5e+03 1.0e-01]
 [0.0e+00 1.6e+03]]


## Manifold Operations

Like other calibration classes in GTSAM, `Cal3_S2Stereo` is treated as a manifold type, even though it is simply a 6-dimensional vector in $\mathbb{R}^6$. This abstraction allows it to integrate seamlessly into GTSAM's optimization framework.

- `retract(d)` maps a small update vector `d` in the tangent space to a new calibration object on the manifold.
- `localCoordinates(T2)` computes the tangent-space vector that moves from the current calibration to another calibration `T2`.

In [16]:
print("Original cal_model:")
print(cal)

# Retract: Apply a delta to the calibration parameters
delta_vec = np.array([10.0, 20.0, 0.05, 1.0, -1.0, 0.01])
cal_retracted = cal.retract(delta_vec)
print(f"\nDelta vector: {delta_vec}")
print("\nRetracted cal_retracted:")
print(cal_retracted)

# Local coordinates: Find the delta between two calibrations
local_coords = cal.localCoordinates(cal_retracted)
print("\nLocal coordinates from cal_model to cal_retracted:")
print(local_coords)

Original cal_model:
K: 1500  0.1  320
   0 1600  240
   0    0    1
Baseline: 0.5


Delta vector: [ 1.e+01  2.e+01  5.e-02  1.e+00 -1.e+00  1.e-02]

Retracted cal_retracted:
K: 1510 0.15  321
   0 1620  239
   0    0    1
Baseline: 0.51


Local coordinates from cal_model to cal_retracted:
[ 1.e+01  2.e+01  5.e-02  1.e+00 -1.e+00  1.e-02]


## Relationship with `StereoCamera`

The `Cal3_S2Stereo` object should be used with the `StereoCamera` class. A `StereoCamera` combines a `Pose3` (the pose of the left camera) and a `Cal3_S2Stereo` calibration to project 3D points into the stereo image plane, producing a `StereoPoint2` which contains the left and right image coordinates ($u_L, u_R, v$). Below is a basic example; please look at the user guide of [`StereoCamera`](./StereoCamera.ipynb) for a more thorough explanation.

In [17]:
fx, fy, s, u0, v0, b = 1000, 1000, 0, 640, 360, 0.5
stereo_calibration = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

# Define the pose of the stereo camera rig in the world frame (left camera's pose)
camera_pose = Pose3(gtsam.Rot3.Ypr(-np.pi/2, 0, -np.pi/2), Point3(0, 0, 5.0))

# Create a StereoCamera object
stereo_camera = StereoCamera(camera_pose, stereo_calibration)

# Define a 3D point in the world frame
p_world = Point3(5, 3, 2)

# Project the 3D point into the stereo camera
p_stereo = stereo_camera.project(p_world)

print(f"3D Point in world frame: {p_world}")
print(f"StereoCamera pose:\n{stereo_camera.pose()}")
print(f"Stereo calibration:\n{stereo_camera.calibration()}")
print(f"\nProjected StereoPoint2 (u_L, u_R, v): {p_stereo}")


3D Point in world frame: [5. 3. 2.]
StereoCamera pose:
R: [
	6.12323e-17, 6.12323e-17, 1;
	-1, 3.7494e-33, 6.12323e-17;
	-0, -1, 6.12323e-17
]
t: 0 0 5

Stereo calibration:
K: 1000    0  640
   0 1000  360
   0    0    1
Baseline: 0.5


Projected StereoPoint2 (u_L, u_R, v): (40, -60, 960)



## Additional Resources

The following resources provide more detailed explanations of camera calibration and stereo vision.

### Article(s)
- ["5.3. Cameras for Robot Vision"](https://www.roboticsbook.org/S53_diffdrive_sensing.html) by Dr. Frank Dellaert and Dr. Seth Hutchinson

### Video(s)
- ["Simple Stereo | Camera Calibration"](https://youtu.be/hUVyDabn1Mg?si=3zIrPPML-H7Zu5_i) by Dr. Shree Nayar from Columbia University

## Source
- [Cal3_S2Stereo.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.h)
- [Cal3_S2Stereo.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.cpp)